### look at the training data

train_v2 = the same equation pairs as v1, re-rendered as stories.
the model sees the story as a prompt and the two RG lines as the answer.

rebuild with `python3 build_train_v2.py` (deterministic — same shas every time).

In [ ]:
import json, random
from pathlib import Path

FT = Path.cwd().parent          # ft-experiments/
rows = [json.loads(l) for l in open(FT / "train_v2/train.jsonl")]
hold = [json.loads(l) for l in open(FT / "train_v2/holdout.jsonl")]
len(rows), len(hold)

In [ ]:
rows[0].keys()

one full example — this is exactly what gets trained on

In [ ]:
r = rows[7]
print(r["story"])
print("\n--- answer ---")
print(r["completion"])

In [ ]:
# a hard one (more ops = deeper nesting)
r = max(rows, key=lambda x: x["ops_total"])
print(r["tier"], r["ops_total"], "ops")
print(r["completion"])

### tiers and themes

`tea` is held out of training on purpose — it's the input-side generalization check.

In [ ]:
from collections import Counter
Counter(r["tier"] for r in rows), Counter(r["theme"] for r in rows)

### disjointness

nothing we train on may appear in the eval set. this is the check that matters most —
hashes are computed under the grader's symmetry group (renaming, side swap, dualization).

In [ ]:
eval_rows = []
for tier in ("normal", "hard", "extra_hard", "order5"):
    eval_rows += [json.loads(l) for l in open(FT / f"eval_v1/eval_{tier}.jsonl")]

train_pairs_ = {r["pair_hash"] for r in rows}
eval_pairs = {r["pair_hash"] for r in eval_rows}
print("train pairs:", len(train_pairs_), " eval pairs:", len(eval_pairs))
print("overlap:", len(train_pairs_ & eval_pairs))     # must be 0

### the eval set

frozen. 777 problems, 4 tiers. same story rendering, plus a literal-NL version.

In [ ]:
e = eval_rows[0]
print(e["tier"], e["problem_id"])
print(e["story"][:400], "...")
print("\n--- reference answer ---")
print(e["reference_rg"])

In [ ]:
# the literal arm — same problem, no story
print(e["literal"][:500])

### the other grammars

we ask for the same answer in notations the model never trained on.
b_near = same shape, different words. b_far = infix, different structure.

In [ ]:
import sys
sys.path.insert(0, str(FT / "eval"))
import grammars
from checkform import parse_prefix_equation

lines = e["reference_rg"].splitlines()
pair_e = parse_prefix_equation(lines[0].split("ASSUME:")[1])
pair_f = parse_prefix_equation(lines[1].split("ASK:")[1])

for g in ("a", "b_near", "b_far"):
    print(f"--- {g} ---")
    print(grammars.GRAMMARS[g].serialize_pair(pair_e, pair_f), "\n")

In [ ]:
# grading is mechanical: parse it back, compare canonical forms
ans = grammars.GRAMMARS["b_far"].serialize_pair(pair_e, pair_f)
grammars.grade_b(ans, "b_far", e["canonical_e"], e["canonical_f"])["status"]